# 🧠 Steering Diffusion Language Models via SAE Feature Intervention

**Extending DLM-Scope (ICLR 2026) for Controllable Reasoning**

Self-contained notebook — no external imports needed. Run all cells in order.

**Runtime**: Change to T4 GPU → `Runtime → Change runtime type → T4 GPU`

In [ ]:
!pip install -q transformers accelerate datasets scipy seaborn safetensors tqdm
import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f}GB')

In [ ]:
# ============ Top-K SAE (DLM-Scope) ============
import torch, torch.nn as nn, torch.nn.functional as F, json
from pathlib import Path

class TopKSAE(nn.Module):
    def __init__(self, d_model, d_dict, k=64):
        super().__init__()
        self.d_model, self.d_dict, self.k = d_model, d_dict, k
        self.encoder = nn.Linear(d_model, d_dict, bias=True)
        self.decoder = nn.Linear(d_dict, d_model, bias=True)
        self.pre_bias = nn.Parameter(torch.zeros(d_model))
        with torch.no_grad(): self.decoder.weight.data = F.normalize(self.decoder.weight.data, dim=0)
        self.feature_activation_counts = torch.zeros(d_dict, dtype=torch.long)

    def encode(self, x):
        h = F.relu(self.encoder(x - self.pre_bias))
        vals, idx = h.topk(self.k, dim=-1)
        out = torch.zeros_like(h)
        out.scatter_(-1, idx, vals)
        return out

    def decode(self, h): return self.decoder(h) + self.pre_bias

    def forward(self, x):
        h = self.encode(x); x_hat = self.decode(h)
        loss = F.mse_loss(x_hat, x)
        l0 = (h > 0).float().sum(-1).mean()
        ev = 1.0 - (x - x_hat).pow(2).sum() / ((x - x.mean(0)).pow(2).sum() + 1e-8)
        active = (h > 0).any(dim=0) if h.dim() == 2 else (h > 0).any(0).any(0)
        self.feature_activation_counts += active.long().cpu()
        return x_hat, h, {'recon_loss': loss, 'l0': l0, 'explained_variance': ev}

    def get_feature_directions(self, indices): return self.decoder.weight[:, indices].detach()
    @property
    def num_dead_features(self): return int((self.feature_activation_counts == 0).sum())
    def _normalize_decoder(self):
        with torch.no_grad(): self.decoder.weight.data = F.normalize(self.decoder.weight.data, dim=0)
    def save(self, path):
        p = Path(path); p.mkdir(parents=True, exist_ok=True)
        torch.save(self.state_dict(), p/'sae.pt')
        json.dump({'d_model':self.d_model,'d_dict':self.d_dict,'k':self.k}, open(p/'cfg.json','w'))

print('✅ TopKSAE defined')

In [ ]:
# ============ DLM Wrapper with DiffuGPT Weight Remapping ============
from transformers import AutoTokenizer, AutoModelForCausalLM
from safetensors.torch import load_file as load_safetensors
from huggingface_hub import hf_hub_download
import logging
logging.basicConfig(level=logging.INFO)

class DLMWrapper:
    def __init__(self, model_name='diffusionfamily/diffugpt-m',
                 base_model_name='gpt2-medium', cache_dir=None):
        self.model_name = model_name
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.cache_dir = cache_dir
        self._activations, self._hooks = {}, []

        # 1. Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name, cache_dir=cache_dir)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        # 2. Load GPT-2 Medium shell
        print('Loading GPT-2 Medium architecture...')
        self.model = AutoModelForCausalLM.from_pretrained(
            base_model_name, cache_dir=cache_dir,
            torch_dtype=torch.float16 if self.device.type == 'cuda' else torch.float32)

        # 3. Resize to 50258 (DiffuGPT = GPT-2 vocab + 1 mask token)
        self.model.resize_token_embeddings(50258)
        self.mask_token_id = 50257

        # 4. Load & remap DiffuGPT weights
        print('Downloading & remapping DiffuGPT weights...')
        ckpt = hf_hub_download(model_name, 'model.safetensors', cache_dir=cache_dir)
        raw = load_safetensors(ckpt)
        mapped = {}
        for k, v in raw.items():
            if k.startswith('denoise_model.'):
                mapped['transformer.' + k[len('denoise_model.'):]] = v
            elif k == 'embed_tokens.weight':
                mapped['transformer.wte.weight'] = v
                mapped['lm_head.weight'] = v.clone()
            else:
                mapped[k] = v
        r = self.model.load_state_dict(mapped, strict=False)
        print(f'  ✅ {len(mapped)} params mapped, {len(r.missing_keys)} missing, {len(r.unexpected_keys)} unexpected')

        # 5. ★ CRITICAL: Disable causal mask → DiffuGPT needs BIDIRECTIONAL attention ★
        for block in self.model.transformer.h:
            block.attn.bias.fill_(1)  # 1 = attend to all positions (no causal mask)
        print('  ✅ Causal mask disabled (bidirectional attention enabled)')

        self.model = self.model.to(self.device)
        self.model.eval()
        self.config = self.model.config
        self.d_model = self.config.n_embd   # 1024
        self.n_layers = self.config.n_layer  # 24
        print(f'✅ DiffuGPT ready: d={self.d_model}, layers={self.n_layers}, device={self.device}')

    def _get_layer(self, idx): return self.model.transformer.h[idx]

    def register_activation_hooks(self, layers):
        self.clear_hooks()
        for idx in layers:
            def mk(i):
                def fn(m, inp, out): self._activations[i] = (out[0] if isinstance(out, tuple) else out).detach().cpu()
                return fn
            self._hooks.append(self._get_layer(idx).register_forward_hook(mk(idx)))

    def clear_hooks(self):
        for h in self._hooks: h.remove()
        self._hooks, self._activations = [], {}

    def get_activations(self): return dict(self._activations)

    @torch.no_grad()
    def forward_pass(self, input_ids, attention_mask=None):
        ids = input_ids.to(self.device)
        mask = (attention_mask if attention_mask is not None else torch.ones_like(ids)).to(self.device)
        return self.model(input_ids=ids, attention_mask=mask).logits

    @torch.no_grad()
    def denoising_loop(self, input_ids, num_steps=30, prefix_len=0, temperature=1.0,
                       steering_hook=None, steering_layer=None,
                       collect_activations=False, activation_layers=None, activation_timesteps=None):
        B, L = input_ids.shape
        gen_len = L - prefix_len
        cur = input_ids.clone().to(self.device)
        mask = torch.zeros(B, L, dtype=torch.bool, device=self.device)
        mask[:, prefix_len:] = True
        cur[:, prefix_len:] = self.mask_token_id
        traj, all_acts = [], {} if collect_activations else None
        if collect_activations and activation_layers: self.register_activation_hooks(activation_layers)
        if activation_timesteps is None: activation_timesteps = list(range(num_steps))

        for step in range(num_steps):
            n_unmask = min(int(gen_len * (step + 1) / num_steps), gen_len)
            sh = None
            if steering_hook and steering_layer is not None:
                lm = self._get_layer(steering_layer)
                def _s(mod, inp, out, _h=steering_hook, _m=mask):
                    return (_h(out[0], _m),) + out[1:] if isinstance(out, tuple) else _h(out, _m)
                sh = lm.register_forward_hook(_s)
            logits = self.forward_pass(cur)
            if sh: sh.remove()
            if collect_activations and step in activation_timesteps:
                all_acts[step] = {l: a.clone() for l, a in self._activations.items()}
            if temperature > 0:
                probs = F.softmax(logits / temperature, dim=-1)
                pred = torch.multinomial(probs.view(-1, probs.size(-1)), 1).view(B, L)
            else:
                pred = logits.argmax(-1)
            conf = F.softmax(logits, -1).gather(-1, pred.unsqueeze(-1)).squeeze(-1)
            filled = cur.clone(); filled[mask] = pred[mask]
            if step < num_steps - 1:
                gc = conf[:, prefix_len:].clone(); gc[~mask[:, prefix_len:]] = float('inf')
                _, keep = gc.topk(n_unmask, dim=-1)
                nm = torch.ones(B, L, dtype=torch.bool, device=self.device); nm[:, :prefix_len] = False
                for b in range(B): nm[b, prefix_len + keep[b]] = False
                cur = filled.clone(); cur[nm] = self.mask_token_id; mask = nm
            else:
                cur = filled; mask = torch.zeros_like(mask)
            traj.append(cur.cpu().clone())
        if collect_activations: self.clear_hooks()
        return {'output_ids': cur.cpu(), 'trajectory': traj, 'activations': all_acts}

    def generate(self, prompt, max_new_tokens=128, num_steps=30, temperature=1.0,
                 steering_hook=None, steering_layer=None):
        pids = self.tokenizer.encode(prompt, return_tensors='pt')
        plen = pids.shape[1]
        ids = torch.cat([pids, torch.full((1, max_new_tokens), self.mask_token_id, dtype=torch.long)], 1)
        r = self.denoising_loop(ids, num_steps, plen, temperature, steering_hook, steering_layer)
        return self.tokenizer.decode(r['output_ids'][0], skip_special_tokens=True)

    def get_model_info(self):
        return {'d_model': self.d_model, 'n_layers': self.n_layers, 'device': str(self.device)}

print('✅ DLMWrapper defined')

In [ ]:
# ============ CELL 4: Load Model & Test ============
import os
SAVE_DIR = '/content/dlm_results'
os.makedirs(SAVE_DIR, exist_ok=True)

dlm = DLMWrapper(cache_dir=os.path.join(SAVE_DIR, 'cache'))

print('\n🧪 Generation test:')
for prompt in ['The capital of France is', 'To solve 3 + 5, we']:
    out = dlm.generate(prompt, max_new_tokens=40, num_steps=20, temperature=0.7)
    print(f'  "{prompt}" → {out[len(prompt):][:100]}')

print('\n🧪 Activation test:')
dlm.register_activation_hooks([4, 12, 20])
_ = dlm.forward_pass(dlm.tokenizer.encode('test input', return_tensors='pt'))
for l, a in dlm.get_activations().items(): print(f'  Layer {l}: {a.shape}')
dlm.clear_hooks()
print('\n✅ Phase 1 complete')

In [ ]:
# ============ CELL 5: GSM8K Data ============
from datasets import load_dataset
import re

class GSM8KLoader:
    COT = 'Solve this math problem step by step.\nQuestion: {q}\nSolution:'
    DIRECT = 'What is the numerical answer?\nQuestion: {q}\nAnswer:'
    def __init__(self, n=200, split='test'):
        ds = load_dataset('openai/gsm8k', 'main', split=split)
        self.problems = list(ds.select(range(min(n, len(ds)))))
        print(f'✅ GSM8K: {len(self.problems)} problems')
    def get_cot(self, idx=None):
        ps = [self.problems[i] for i in idx] if idx else self.problems
        return [{'prompt': self.COT.format(q=p['question']), 'answer': self._ans(p['answer'])} for p in ps]
    def get_direct(self, idx=None):
        ps = [self.problems[i] for i in idx] if idx else self.problems
        return [{'prompt': self.DIRECT.format(q=p['question']), 'answer': self._ans(p['answer'])} for p in ps]
    @staticmethod
    def _ans(t):
        m = re.search(r'####\s*([\d,\.\-]+)', t)
        return m.group(1).replace(',','') if m else '0'

gsm8k = GSM8KLoader(n=200)
disc_idx, eval_idx = list(range(100)), list(range(100, 200))
print(f'Discovery: {len(disc_idx)}, Eval: {len(eval_idx)}')
print(f'\nSample: {gsm8k.get_cot([0])[0]["prompt"][:150]}...')

In [ ]:
# ============ CELL 6: Collect Activations ============
from tqdm import tqdm
import gc

LAYERS = [4, 8, 12, 16, 20]
N_DISC = 50
STEPS = 20

def collect(dlm, prompts, layers, n, steps=20):
    acts = {l: [] for l in layers}
    for i, p in enumerate(tqdm(prompts[:n], desc='Collecting')):
        pids = dlm.tokenizer.encode(p['prompt'], return_tensors='pt', max_length=128, truncation=True)
        plen = pids.shape[1]
        ids = torch.cat([pids, torch.full((1, 64), dlm.mask_token_id, dtype=torch.long)], 1)
        r = dlm.denoising_loop(ids, steps, plen, 0.8, collect_activations=True,
                               activation_layers=layers, activation_timesteps=[steps//2])
        mid = steps // 2
        if r['activations'] and mid in r['activations']:
            for l in layers:
                if l in r['activations'][mid]:
                    acts[l].append(r['activations'][mid][l].float().mean(dim=1))
        if (i+1) % 10 == 0: gc.collect(); torch.cuda.empty_cache()
    return {l: torch.cat(a, 0) for l, a in acts.items() if a}

print('📊 CoT activations...')
cot_acts = collect(dlm, gsm8k.get_cot(disc_idx), LAYERS, N_DISC, STEPS)
print(f'  CoT shapes: {{", ".join(f"L{l}:{a.shape}" for l,a in cot_acts.items())}}')

print('📊 Direct activations...')
direct_acts = collect(dlm, gsm8k.get_direct(disc_idx), LAYERS, N_DISC, STEPS)
print(f'  Direct shapes: {{", ".join(f"L{l}:{a.shape}" for l,a in direct_acts.items())}}')

torch.save({'cot': cot_acts, 'direct': direct_acts}, f'{SAVE_DIR}/activations.pt')
print('\n✅ Phase 2 complete')

In [ ]:
# ============ CELL 7: Train SAE ============
from torch.utils.data import DataLoader, TensorDataset

LYR = 16  # DLM-Scope: deep layers best for steering
train_data = torch.cat([cot_acts[LYR], direct_acts[LYR]], 0)
d_model, d_dict, K = train_data.shape[1], 4 * train_data.shape[1], 64

sae = TopKSAE(d_model, d_dict, K).to(dlm.device)
opt = torch.optim.Adam(sae.parameters(), lr=3e-4)
loader = DataLoader(TensorDataset(train_data), batch_size=32, shuffle=True)

for ep in range(10):
    tl, te, n = 0, 0, 0
    for (b,) in loader:
        _, _, m = sae(b.to(dlm.device))
        opt.zero_grad(); m['recon_loss'].backward()
        torch.nn.utils.clip_grad_norm_(sae.parameters(), 1.0)
        opt.step(); sae._normalize_decoder()
        tl += m['recon_loss'].item(); te += m['explained_variance'].item(); n += 1
    print(f'  Epoch {ep+1}: loss={tl/n:.4f}, EV={te/n:.3f}, dead={sae.num_dead_features}/{d_dict}')

sae.eval(); sae.save(f'{SAVE_DIR}/sae')
print(f'\n✅ Phase 3: SAE trained (EV={te/n:.3f})')

In [ ]:
# ============ CELL 8: Contrastive Feature Discovery (NOVEL) ============
import numpy as np
from scipy import stats

sae.eval()
with torch.no_grad():
    cf = sae.encode(cot_acts[LYR].to(dlm.device)).cpu().numpy()
    df = sae.encode(direct_acts[LYR].to(dlm.device)).cpu().numpy()

diffs, pvals, cohens = [], [], []
for f in range(d_dict):
    c, d = cf[:, f], df[:, f]
    diff = c.mean() - d.mean()
    cs, ds = c.std()+1e-8, d.std()+1e-8
    _, p = stats.ttest_ind(c, d, equal_var=False) if (cs > 1e-7 or ds > 1e-7) else (0, 1.0)
    cd = diff / (np.sqrt((cs**2 + ds**2)/2) + 1e-8)
    diffs.append(diff); pvals.append(p/2 if diff > 0 else 1.0); cohens.append(cd)

diffs, pvals, cohens = np.array(diffs), np.array(pvals), np.array(cohens)
sig = pvals < (0.05 / d_dict)  # Bonferroni
reasoning_mask = sig & (np.abs(cohens) >= 0.2) & (diffs > 0)
reasoning_idx = np.where(reasoning_mask)[0]
reasoning_features = reasoning_idx[np.argsort(cohens[reasoning_idx])[::-1]].tolist()[:50]

if not reasoning_features:  # Fallback: top by effect size
    print('⚠️ No Bonferroni-significant. Using top by effect size.')
    reasoning_features = [int(i) for i in np.argsort(cohens)[::-1][:50] if diffs[i] > 0][:50]

print(f'🔬 Found {len(reasoning_features)} reasoning features')
print(f'   Top 10: {reasoning_features[:10]}')
print('✅ Phase 4 complete')

In [ ]:
# ============ CELL 9: Steering Experiments ============
import random

class Steerer:
    def __init__(self, sae, feats, alpha=2.0):
        d = sae.get_feature_directions(feats).sum(1)  # [d_model]
        self.dir = d / (d.norm() + 1e-8)
        self.alpha = alpha
    def hook(self):
        d, a = self.dir, self.alpha
        def fn(hs, mask): return hs + a * d.to(hs.device).unsqueeze(0).unsqueeze(0)
        return fn

def run_exp(dlm, prompts, sae=None, feats=None, layer=None, alpha=0.0, label='', n=20):
    h, hl = (Steerer(sae, feats, alpha).hook(), layer) if (alpha != 0 and feats) else (None, None)
    results = []
    for p in tqdm(prompts[:n], desc=label):
        t = dlm.generate(p['prompt'], max_new_tokens=128, num_steps=20, temperature=0.7,
                         steering_hook=h, steering_layer=hl)
        results.append({'prompt': p['prompt'], 'generated': t, 'answer': p.get('answer','')})
    return results

eval_cot = gsm8k.get_cot(eval_idx)
N = 20

print('[E1] Baseline...')
baseline = run_exp(dlm, eval_cot, n=N, label='baseline')
print('[E2] Positive (α=2)...')
pos = run_exp(dlm, eval_cot, sae, reasoning_features, LYR, 2.0, 'pos_2.0', N)
print('[E3] Negative (α=-2)...')
neg = run_exp(dlm, eval_cot, sae, reasoning_features, LYR, -2.0, 'neg_-2.0', N)
print('[E4] Random control...')
random.seed(42)
rnd = run_exp(dlm, eval_cot, sae, random.sample(range(d_dict), 50), LYR, 2.0, 'random', N)
print('✅ Phase 5 complete')

In [ ]:
# ============ CELL 10: Evaluation ============
def extract_num(t):
    m = re.search(r'####\s*([\d,\.\-]+)', t)
    if m: return m.group(1).replace(',','')
    m = re.search(r'answer\s+is\s+([\d,\.\-]+)', t, re.I)
    if m: return m.group(1).replace(',','')
    ns = re.findall(r'-?[\d,]+\.?\d*', t)
    return ns[-1].replace(',','') if ns else None

def rscore(t):
    s = sum(1 for w in ['step','first','then','therefore','because','total','calculate'] if w in t.lower())
    s += len(re.findall(r'[+\-*/=]', t)) * 0.5
    s += len(re.findall(r'\d+', t)) * 0.3
    s += len(re.findall(r'\d+\s*[+\-*/]\s*\d+', t)) * 2.0
    return s

def evaluate(results, label):
    correct = sum(1 for r in results if extract_num(r['generated']) and r['answer']
                  and abs(float(extract_num(r['generated']) or 0) - float(r['answer'])) < 0.01)
    scores = [rscore(r['generated']) for r in results]
    return {'label': label, 'acc': correct/len(results), 'correct': correct,
            'total': len(results), 'rscore': np.mean(scores)}

print('='*65)
print(f'{"Condition":<25} {"Accuracy":<12} {"Reasoning":<12} {"Correct"}')
print('-'*65)
evals = {}
for res, lbl in [(baseline,'Baseline'),(pos,'Positive α=2'),(neg,'Negative α=-2'),(rnd,'Random Ctrl')]:
    e = evaluate(res, lbl); evals[lbl] = e
    print(f'{lbl:<25} {e["acc"]:>8.1%}     {e["rscore"]:>8.1f}      {e["correct"]}/{e["total"]}')
print('='*65)

print('\n📝 Samples:')
for i in range(min(3, len(baseline))):
    print(f'\n--- Q{i+1} ---')
    print(f'Baseline:  {baseline[i]["generated"][len(baseline[i]["prompt"]):][: 120]}')
    print(f'Steered+:  {pos[i]["generated"][len(pos[i]["prompt"]):][: 120]}')
    print(f'Expected:  {baseline[i]["answer"]}')

In [ ]:
# ============ CELL 11: Alpha Sweep ============
alpha_evals = []
for a in [0.5, 1.0, 2.0, 3.0, 5.0]:
    r = run_exp(dlm, eval_cot, sae, reasoning_features, LYR, a, f'α={a}', 15)
    e = evaluate(r, f'α={a}'); alpha_evals.append({'alpha': a, **e})
    print(f'  α={a}: acc={e["acc"]:.1%}, reasoning={e["rscore"]:.1f}')
print('✅ Sweep done')

In [ ]:
# ============ CELL 12: Visualization ============
import matplotlib.pyplot as plt, seaborn as sns
plt.style.use('dark_background')
C = {'pos': '#00d4aa', 'neg': '#ff6b6b', 'rnd': '#ffd93d', 'base': '#6c8ebf'}
os.makedirs(f'{SAVE_DIR}/figs', exist_ok=True)

# Fig 1: Top features
fig, ax = plt.subplots(figsize=(12,5))
top = reasoning_features[:25]
ax.barh(range(len(top)), [diffs[f] for f in top], color=[plt.cm.viridis(cohens[f]/max(cohens[top]+1e-8)) for f in top])
ax.set_yticks(range(len(top))); ax.set_yticklabels([f'F{f}' for f in top], fontsize=8)
ax.set_xlabel('Differential Activation (CoT − Direct)'); ax.set_title('Top Reasoning SAE Features', fontweight='bold')
ax.invert_yaxis(); plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/figs/fig1.png', dpi=150); plt.show()

# Fig 2: Results
fig, (a1,a2) = plt.subplots(1,2,figsize=(14,5))
labs = list(evals.keys()); cols = [C['base'],C['pos'],C['neg'],C['rnd']]
a1.bar(range(4), [evals[l]['acc']*100 for l in labs], color=cols)
a1.set_xticks(range(4)); a1.set_xticklabels(labs, fontsize=9); a1.set_ylabel('Accuracy %'); a1.set_title('GSM8K Accuracy', fontweight='bold')
a2.bar(range(4), [evals[l]['rscore'] for l in labs], color=cols)
a2.set_xticks(range(4)); a2.set_xticklabels(labs, fontsize=9); a2.set_ylabel('Score'); a2.set_title('Reasoning Quality', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/figs/fig2.png', dpi=150); plt.show()

# Fig 3: Alpha sweep
if alpha_evals:
    fig, ax = plt.subplots(figsize=(8,5))
    ax.plot([e['alpha'] for e in alpha_evals], [e['acc']*100 for e in alpha_evals], 'o-', color=C['pos'], lw=2, ms=8)
    ax.axhline(evals['Baseline']['acc']*100, color=C['base'], ls='--', alpha=.7, label='Baseline')
    ax.set_xlabel('Steering Strength (α)'); ax.set_ylabel('Accuracy %'); ax.set_title('Accuracy vs α', fontweight='bold')
    ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/figs/fig3.png', dpi=150); plt.show()

print(f'📊 Figures saved to {SAVE_DIR}/figs/')

In [ ]:
# ============ CELL 13: Save Everything ============
print('='*60)
print('🎉 FINAL RESULTS')
print('='*60)
for l, e in evals.items(): print(f'{l:<25} acc={e["acc"]:.1%}  reasoning={e["rscore"]:.1f}')
if alpha_evals:
    print('\nAlpha sweep:')
    for e in alpha_evals: print(f'  α={e["alpha"]}: acc={e["acc"]:.1%}, score={e["rscore"]:.1f}')

json.dump({'evals': evals, 'alpha': alpha_evals, 'features': reasoning_features,
           'config': {'d_model': d_model, 'd_dict': d_dict, 'k': K, 'layer': LYR}},
          open(f'{SAVE_DIR}/results.json','w'), indent=2, default=str)
print(f'\n💾 Saved to {SAVE_DIR}/')
print('🏁 Pipeline complete!')